<div style="font-size: 0.95em;">
  <h3>What We'll Cover in This Notebook</h3>
  <ol>
    <li><strong>Basic <code>ChatPromptTemplate</code></strong><br>
        Using static messages (system, human) and placeholders like <code>{topic}</code> for dynamic values.
    </li>
    <li><strong>Multiple Placeholders</strong><br>
        Using several variables in one template and passing a dictionary with multiple values.
    </li>
    <li><strong>Partial Variables</strong><br>
        Pre-filling some values so they don't need to be passed each time using <code>.partial()</code>.
    </li>
    <li><strong><code>MessagesPlaceholder</code></strong> (preview for chat history)<br>
        Using a placeholder for a list of messages; will be used later for conversational RAG.
    </li>
    <li><strong>Few-Shot Prompting</strong> (later in Module 2 Part B)<br>
        Using examples to guide the model with <code>FewShotChatMessagePromptTemplate</code>.
    </li>
  </ol>
</div>

Basic ChatPromptTemplate – Concept
* A ChatPromptTemplate is a structure that defines a list of messages (system, human, etc.) with placeholders for values that change per request.

Example of template:
* System: You are a helpful assistant.
* Human: Explain {topic} in one sentence.

The {topic} is a placeholder. When you call the template with topic="RAG", it fills in the placeholder.
* Placeholders are defined using curly braces {variable_name}.

In [1]:
# Step 1: Imports and Environment
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

In [2]:
# Load environment variables from .env
load_dotenv()

True

Step 2: Create a Basic Prompt Template with a Placeholder

Create a template that has:

* A system message (static)

* A human message with a {topic} placeholder

In [5]:
# Create a ChatPromptTemplate with a placeholder {topic}
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Explain {topic} in one sentence.')
])

# Format the template with a value for {topic}
formatted_messages = prompt.format_messages(topic='RAG')

for message in formatted_messages:
    print(f'[{message.type}] {message.content}')

[system] You are a helpful assistant.
[human] Explain RAG in one sentence.


<div style="font-size: 0.85em;">
  <h3><code>format_messages()</code> vs <code>.invoke()</code></h3>
  <ul>
    <li><strong><code>format_messages()</code></strong><br>
        Replaces placeholders with actual values and returns formatted message objects.<br>
        <em>Does NOT call the LLM.</em> Use it to inspect the prompt before sending.
    </li>
    <li><strong><code>.invoke()</code></strong><br>
        Takes input, calls the LLM, and returns the AI's response.<br>
        <em>Does call the LLM.</em> Use it to get actual answers.
    </li>
  </ul>
  <p>When you run <code>prompt | llm</code> and call <code>invoke()</code>, LangChain automatically:<br>
  1) calls <code>format_messages()</code> internally,<br>
  2) passes the formatted messages to the LLM,<br>
  3) returns the model's response.</p>
  <p><strong>Analogy:</strong> <code>format_messages()</code> is like writing a letter with blanks filled in, but not mailing it. <code>invoke()</code> is like mailing the letter and receiving a reply.</p>
</div>

Step 3: Create a Template with Multiple Placeholders

In [7]:
prompt_multi = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Explain {topic} to a {audience} in {word_count} words.')
])

# Format the template with values for all placeholders
formatted_messages = prompt_multi.format_messages(
    topic='RAG',
    audience='beginner',
    word_count='100'
)

# Print each formatted message
for message in formatted_messages:
    print(f'[{message.type}] {message.content}')

[system] You are a helpful assistant.
[human] Explain RAG to a beginner in 100 words.


Partial Variables – Concept

Why use partial variables?

* Reduces repetition when many calls share the same values.

* Makes the template easier to reuse.

Step 4: Create a Template with a Partial Variable

In [10]:
# Create a template with two placeholders: {topic} and {audience}
prompt_partial = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Explain {topic} to a {audience}.')
])

# Pre-fill {audience} with 'beginner'
prompt_partial = prompt_partial.partial(audience='beginner')

# Format the template  to provide only {topic}
formatted_messages = prompt_partial.format_messages(topic='RAG')

for message in formatted_messages:
    print(f'[{message.type}] {message.content}')

[system] You are a helpful assistant.
[human] Explain RAG to a beginner.


#### MessagesPlaceholder – Concept
* MessagesPlaceholder is used when we want to insert a list of messages into the prompt, rather than a single value. This is important for conversational RAG, where we need to include the chat history.

How it works:
* A MessagesPlaceholder named history can be filled with a list of HumanMessage, AIMessage, etc.
* The placeholder will be replaced by those messages in the order provided.



Step 5: Using MessagesPlaceholder

In [16]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Create a template with a placeholder for history
prompt_with_history = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', 'What is the capital of Nigeria?')
])

# Define a sample chat history (list of messages)
chat_history = [
    HumanMessage(content='Hi, my name is Saheed.'),
    AIMessage(content='Hi Saheed, how can I help you?')
]

# Format the template with the history
formatted_messages = prompt_with_history.format_messages(history=chat_history)

# Loop through messages
for msg in formatted_messages:
    print(f'[{msg.type}] {msg.content}')

[system] You are a helpful assistant.
[human] Hi, my name is Saheed.
[ai] Hi Saheed, how can I help you?
[human] What is the capital of Nigeria?


<div style="font-size: 0.85em;">
  <h3>Summary: Prompt Templates</h3>
  <ul>
    <li><strong>Basic <code>ChatPromptTemplate</code></strong> – define system and human messages with static text and placeholders (<code>{topic}</code>).</li>
    <li><strong>Multiple placeholders</strong> – supply a dictionary with all keys when formatting/invoking.</li>
    <li><strong>Partial variables</strong> – use <code>.partial()</code> to pre-fill values, so they don't need to be passed each time.</li>
    <li><strong><code>MessagesPlaceholder</code></strong> – insert a list of messages (e.g., chat history) into the prompt. Essential for conversational RAG.</li>
    <li><strong><code>format_messages()</code></strong> only prepares the messages; <strong><code>invoke()</code></strong> calls the LLM and returns a response.</li>
  </ul>
</div>